<a href="https://colab.research.google.com/github/angulorojasmariaclaudia-coder/PRACTICAS-DE-APRENDIZAJE-PROFUNDO/blob/main/Practica10_Desarrollo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



```
Práctica N°10 - Aprendizaje Profundo
```



# **INTERPRETABILIDAD DEL MODELO**

## *DataSet: Poem Sentiment*

###00. Librerías

In [1]:
!pip install -q datasets evaluate transformers[sentencepiece] accelerate shap matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


In [2]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import shap

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo detectado:", device)

Dispositivo detectado: cuda


###01. Recuperación del modelo de la práctica 7


In [5]:
MODEL_PATH = "Clau31/poem-sentiment-bert"

print("Cargando modelo desde Hugging Face:", MODEL_PATH)

Cargando modelo desde Hugging Face: Clau31/poem-sentiment-bert


In [9]:
from huggingface_hub import list_repo_files
list_repo_files("Clau31/poem-sentiment-bert")

['.gitattributes',
 'README.md',
 'config.json',
 'model.safetensors',
 'tokenizer.json',
 'tokenizer_config.json']

In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("Clau31/poem-sentiment-bert")
tokenizer = AutoTokenizer.from_pretrained("Clau31/poem-sentiment-bert")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [15]:
dataset = load_dataset("poem_sentiment")

train_ds = dataset["train"]
valid_ds = dataset["validation"]
test_ds = dataset["test"]

print(dataset)
print("Tamaño train:", len(train_ds))
print("Tamaño validation:", len(valid_ds))
print("Tamaño test:", len(test_ds))

label_names = train_ds.features["label"].names
id2label = {i: label_names[i] for i in range(len(label_names))}
label2id = {v: k for k, v in id2label.items()}
print("Etiquetas:", label_names)

pd.DataFrame(train_ds[:5])

DatasetDict({
    train: Dataset({
        features: ['id', 'verse_text', 'label'],
        num_rows: 892
    })
    validation: Dataset({
        features: ['id', 'verse_text', 'label'],
        num_rows: 105
    })
    test: Dataset({
        features: ['id', 'verse_text', 'label'],
        num_rows: 104
    })
})
Tamaño train: 892
Tamaño validation: 105
Tamaño test: 104
Etiquetas: ['negative', 'positive', 'no_impact', 'mixed']


,id,verse_text,label
0,0,with pale blue berries. in these peaceful shad...,1
1,1,"it flows so long as falls the rain,",2
2,2,"and that is why, the lonesome day,",0
3,3,"when i peruse the conquered fame of heroes, an...",3
4,4,of inward strife for truth and liberty.,3


In [25]:
classifier = pipeline(
    "text-classification",
    model="Clau31/poem-sentiment-bert",
    tokenizer="Clau31/poem-sentiment-bert",
    top_k=None
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [26]:
texto_prueba = "This poem is amazing"
pred = classifier(texto_prueba)
print(pred)

[[{'label': 'positive', 'score': 0.5166290998458862}, {'label': 'no_impact', 'score': 0.3275919556617737}, {'label': 'mixed', 'score': 0.09069761633872986}, {'label': 'negative', 'score': 0.06508133560419083}]]


###02. Función auxiliar para obtener probabilidades por clase

In [27]:
labels = ["negative", "positive", "no_impact", "mixed"]

def predict_proba(texts):
    outputs = classifier(list(texts))
    probas = []

    for out in outputs:
        scores = {item["label"]: item["score"] for item in out}
        probas.append([scores[label] for label in labels])

    return np.array(probas)

###03. Creación del explicador SHAP

In [28]:
shap.Explainer(predict_proba, tokenizer, output_names=label_names)
print("Explicador SHAP preparado.")

Explicador SHAP preparado.


###04. Explicación de una predicción individual

In [29]:
sample_text = test_ds[0]["verse_text"]
shap_values_single = explainer([sample_text])

print("Texto analizado:\n")
print(sample_text)

shap.plots.text(shap_values_single[0])

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Texto analizado:

my canoe to make more steady,


###05. Explicación de varias muestras del conjunto de test

In [30]:
num_examples = 5
texts_batch = test_ds["verse_text"][:num_examples]

shap_values_batch = explainer(texts_batch)
shap.plots.text(shap_values_batch)

###06.Función para analizar cualquier ejemplo por índice

In [31]:
def explicar_ejemplo(idx):
    texto = test_ds[idx]["verse_text"]
    etiqueta_real = id2label[test_ds[idx]["label"]]
    predicciones = classifier(texto)[0]
    etiqueta_pred = max(predicciones, key=lambda x: x["score"])["label"]

    print(f"Índice: {idx}")
    print(f"Etiqueta real: {etiqueta_real}")
    print(f"Etiqueta predicha: {etiqueta_pred}")
    print("\nTexto:\n")
    print(texto)

    valores = explainer([texto])
    shap.plots.text(valores[0])

# Ejemplo
explicar_ejemplo(1)

Índice: 1
Etiqueta real: positive
Etiqueta predicha: no_impact

Texto:

and be glad in the summer morning when the kindred ride on their way;


###07. Comparación de varias predicciones del conjunto de test

In [34]:
def resumen_predicciones(indices):
    filas = []
    for idx in indices:
        texto = test_ds[idx]["verse_text"]
        real = id2label[test_ds[idx]["label"]]
        predicciones = classifier(texto)[0]
        mejor = max(predicciones, key=lambda x: x["score"])
        filas.append({
            "idx": idx,
            "texto": texto[:120] + ("..." if len(texto) > 120 else ""),
            "real": real,
            "predicha": mejor["label"],
            "score_pred": round(mejor["score"], 4)
        })
    return pd.DataFrame(filas)

df_resumen = resumen_predicciones([0,1,2,3,4,5,6,7,8,9])
df_resumen

,idx,texto,real,predicha,score_pred
0,0,"my canoe to make more steady,",no_impact,no_impact,0.8833
1,1,and be glad in the summer morning when the kin...,positive,no_impact,0.5756
2,2,and when they reached the strait symplegades,no_impact,no_impact,0.8885
3,3,she sought for flowers,no_impact,no_impact,0.8022
4,4,"if they are hungry, paradise",no_impact,no_impact,0.7187
5,5,indignantly i hurled the cry:,negative,negative,0.5955
6,6,with which his house is haunted;,negative,negative,0.5912
7,7,"and, laying snow-white flowers against my hair.",no_impact,no_impact,0.5446
8,8,"of long-uncoupled bed, and childless eld,",no_impact,no_impact,0.5362
9,9,"of the boulder-strewn mountain, and when they ...",no_impact,no_impact,0.9033
